# Foot Dataset Alignment Playground

This notebook follows the same logic as `foot_axis_debug_playground.ipynb` and `foot_alignment_playground.ipynb`, but it works with the full-dataset debug outputs.

The goal is not training. The goal is to answer, clearly:

```text
1. Which way do the raw SUPR foot axes point?
2. Which way do the raw shoe axes point?
3. Does the SUPR-to-shoe axis remap make sense?
4. Is the whole neutral foot actually inside the shoe?
5. Does the red sole-block surface sit below the aligned foot?
```

Color language:

```text
orange = aligned SUPR foot
gray   = watertight shoe before mSDF cuts
blue   = final/open shoe after mSDF cuts
red    = selected sole-block triangles under the foot
purple = detected shoe opening/collar boundary
```


## 1. Paths And Dataset Scan

The batch script writes reusable outputs under `baselines/GShell/output/foot_alignment_turntable-512-768`.


In [1]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '5')

from pathlib import Path
import csv
import json
import sys

import numpy as np
import torch

FOOTSHELL_ROOT = Path('/data/abelde/projects/active/Shell_Gaussian/FootShellGaussian')
PROJECT_ROOT = FOOTSHELL_ROOT.parent
DATASET_ROOT = Path('/data/abelde/datasets/processed/gshell_shoes_turntable_canonical')
BASELINE_OUTPUT_ROOT = PROJECT_ROOT / 'baselines' / 'GShell' / 'output'
BASELINE_SUBDIR = 'turntable-512-768'
BASELINE_SUFFIX = '_turntable'
GSHELL_CONFIG = PROJECT_ROOT / 'baselines' / 'GShell' / 'configs' / 'shoes_mc_normfix_512_768.json'
DEBUG_ROOT = BASELINE_OUTPUT_ROOT / 'foot_alignment_turntable-512-768'
GSHELL_ENV = PROJECT_ROOT / 'baselines' / 'GShell' / 'GShell_env'
FOOT_OBJ_PATH = PROJECT_ROOT / 'baselines' / 'SUPR' / 'output' / 'debug_playground' / 'supr_male_right_foot_neutral.obj'
FOOT_SDF_PATH = FOOTSHELL_ROOT / 'data' / 'foot_prior' / 'supr_male_right_foot_sdf.npz'

if str(FOOTSHELL_ROOT) not in sys.path:
    sys.path.insert(0, str(FOOTSHELL_ROOT))

shoe_names = sorted(p.name for p in DATASET_ROOT.iterdir() if p.is_dir())
print('dataset shoes:', len(shoe_names))
for name in shoe_names:
    print('  ', name)
print('\ndebug root:', DEBUG_ROOT, '| exists=', DEBUG_ROOT.exists())
print('GShell config:', GSHELL_CONFIG, '| exists=', GSHELL_CONFIG.exists())
print('foot OBJ  :', FOOT_OBJ_PATH, '| exists=', FOOT_OBJ_PATH.exists())
print('foot SDF  :', FOOT_SDF_PATH, '| exists=', FOOT_SDF_PATH.exists())


dataset shoes: 20
   Adidas-Yeezy-Boost-350-V2-Desert-Sage-Infant
   Adidas-Yeezy-Boost-350-V2-Static-Non-Reflective-Infants
   Adidas-Yeezy-Boost-350-V2-Static-Non-Reflective-Kids
   Air-Jordan-1-Mid-Wear-Away-Chicago-Gs
   Air-Jordan-1-Retro-High-Hyper-Royal-Smoke-Grey-Gs
   Air-Jordan-1-Retro-High-Og-Washed-Black-Gs
   Air-Jordan-1-Retro-High-Og-White-Cement-Gs
   Air-Jordan-12-Retro-Arctic-Punch-Gs
   Air-Jordan-13-Retro-Houndstooth-Gs
   Air-Jordan-5-Retro-Plaid-Gs
   Air-Jordan-6-Retro-Washed-Denim-2022-Gs
   Birkenstock-Boston-Suede-Stone-Coin
   Crocs-Classic-Clog-Cinnamon-Toast-Crunch
   Crocs-Classic-Clog-Cinnamon-Toast-Crunch-Gs
   Crocs-Classic-Clog-Cocoa-Puffs-Kids
   Crocs-Classic-Clog-Staple-Sidewalk-Luxe
   Nike-Calm-Slide-Cinnamon-Monarch
   Nike-Cortez-Se-Suede-Pacific-Moss-Infinite-Gold-Muslin-Sail
   Ugg-Bailey-Bow-Ii-Boot-Ribbon-Red-Kids
   Ugg-Classic-Short-Ii-Boot-Rock-Rose-Toddler

debug root: /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output/f

## 2. Generate Or Refresh Debug Outputs

Run the executable cell below when you want to refresh the saved turntable-canonical debug outputs. It builds the same command using `sys.argv`, so you can run the script from inside this notebook. The script uses the already-exported GShell meshes from `turntable-512-768` and their `mesh_watertight` exports rather than regenerating them from `model.pt`.

Selected turntable shoes:

```bash
CUDA_VISIBLE_DEVICES=5 /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/GShell_env/bin/python \
  /data/abelde/projects/active/Shell_Gaussian/FootShellGaussian/scripts/prepare_dataset_foot_alignment_debug.py \
  --dataset-root /data/abelde/datasets/processed/gshell_shoes_turntable_canonical \
  --baseline-output-root /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output \
  --baseline-subdir turntable-512-768 \
  --baseline-suffix _turntable \
  --gshell-config /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/configs/shoes_mc_normfix_512_768.json \
  --out-root /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output/foot_alignment_turntable-512-768 \
  --shoe-name Air-Jordan-1-Mid-Wear-Away-Chicago-Gs \
  --shoe-name Air-Jordan-1-Retro-High-Og-Washed-Black-Gs \
  --shoe-name Air-Jordan-1-Retro-High-Og-White-Cement-Gs \
  --shoe-name Air-Jordan-5-Retro-Plaid-Gs \
  --shoe-name Air-Jordan-12-Retro-Arctic-Punch-Gs \
  --shoe-name Air-Jordan-13-Retro-Houndstooth-Gs \
  --shoe-name Crocs-Classic-Clog-Cinnamon-Toast-Crunch-Gs \
  --shoe-name Ugg-Bailey-Bow-Ii-Boot-Ribbon-Red-Kids \
  --shoe-name Ugg-Classic-Short-Ii-Boot-Rock-Rose-Toddler \
  --overwrite
```


In [3]:
import runpy
import shlex

# Safety guard: set this to True when you actually want to regenerate artifacts.
RUN_PREPARE = True

# Set PREPARE_ALL_SHOES=True only after every scene under DATASET_ROOT has a trained turntable mesh.
PREPARE_ALL_SHOES = False
PREPARE_SHOE_NAMES = [
    'Air-Jordan-1-Mid-Wear-Away-Chicago-Gs',
    'Air-Jordan-1-Retro-High-Og-Washed-Black-Gs',
    'Air-Jordan-1-Retro-High-Og-White-Cement-Gs',
    'Air-Jordan-5-Retro-Plaid-Gs',
    'Air-Jordan-12-Retro-Arctic-Punch-Gs',
    'Air-Jordan-13-Retro-Houndstooth-Gs',
    'Crocs-Classic-Clog-Cinnamon-Toast-Crunch-Gs',
    'Ugg-Bailey-Bow-Ii-Boot-Ribbon-Red-Kids',
    'Ugg-Classic-Short-Ii-Boot-Rock-Rose-Toddler',
]
PREPARE_OVERWRITE = True
PREPARE_FORCE_REEXPORT_FROM_CHECKPOINT = False
PREPARE_DEVICE = 'cuda'

prepare_script = FOOTSHELL_ROOT / 'scripts' / 'prepare_dataset_foot_alignment_debug.py'
prepare_argv = [
    str(prepare_script),
    '--dataset-root', str(DATASET_ROOT),
    '--baseline-output-root', str(BASELINE_OUTPUT_ROOT),
    '--baseline-subdir', BASELINE_SUBDIR,
    '--baseline-suffix', BASELINE_SUFFIX,
    '--gshell-config', str(GSHELL_CONFIG),
    '--out-root', str(DEBUG_ROOT),
    '--device', PREPARE_DEVICE,
]
if not PREPARE_ALL_SHOES:
    for shoe_name in PREPARE_SHOE_NAMES:
        prepare_argv += ['--shoe-name', shoe_name]
if PREPARE_OVERWRITE:
    prepare_argv += ['--overwrite']
if PREPARE_FORCE_REEXPORT_FROM_CHECKPOINT:
    prepare_argv += ['--force-reexport-from-checkpoint']

shell_equivalent = [str(GSHELL_ENV / 'bin' / 'python')] + prepare_argv
print('CUDA_VISIBLE_DEVICES=' + os.environ.get('CUDA_VISIBLE_DEVICES', ''))
print(' '.join(shlex.quote(part) for part in shell_equivalent))

if RUN_PREPARE:
    old_argv = sys.argv[:]
    try:
        sys.argv = prepare_argv
        runpy.run_path(str(prepare_script), run_name='__main__')
    finally:
        sys.argv = old_argv
else:
    print('Set RUN_PREPARE = True, then execute this cell to run prepare_dataset_foot_alignment_debug.py.')


CUDA_VISIBLE_DEVICES=5
/data/abelde/projects/active/Shell_Gaussian/baselines/GShell/GShell_env/bin/python /data/abelde/projects/active/Shell_Gaussian/FootShellGaussian/scripts/prepare_dataset_foot_alignment_debug.py --dataset-root /data/abelde/datasets/processed/gshell_shoes_turntable_canonical --baseline-output-root /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output --baseline-subdir turntable-512-768 --baseline-suffix _turntable --gshell-config /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/configs/shoes_mc_normfix_512_768.json --out-root /data/abelde/projects/active/Shell_Gaussian/baselines/GShell/output/foot_alignment_turntable-512-768 --device cuda --shoe-name Air-Jordan-1-Mid-Wear-Away-Chicago-Gs --shoe-name Air-Jordan-1-Retro-High-Og-Washed-Black-Gs --shoe-name Air-Jordan-1-Retro-High-Og-White-Cement-Gs --shoe-name Air-Jordan-5-Retro-Plaid-Gs --shoe-name Air-Jordan-12-Retro-Arctic-Punch-Gs --shoe-name Air-Jordan-13-Retro-Houndstooth-Gs --shoe-name 

KeyboardInterrupt: 

## 3. Pick One Shoe

This notebook can inspect any shoe that already has debug outputs.


In [ ]:
DEFAULT_SHOE = 'Adidas-Yeezy-Boost-350-V2-Static-Non-Reflective-Kids'
SHOE_NAME = DEFAULT_SHOE

shoe_out = DEBUG_ROOT / SHOE_NAME
summary_path = shoe_out / 'summary.json'
print('selected shoe:', SHOE_NAME)
print('output folder:', shoe_out)
print('summary exists:', summary_path.exists())

if not summary_path.exists():
    raise FileNotFoundError(f'Missing {summary_path}. Run the command in section 2 first.')


## 4. Load Meshes, Foot, SDF, And Saved Alignment

Important difference from the older one-shoe notebooks: here the shoe mesh paths come from the saved batch summary. By default the batch script uses the already-exported GShell training meshes under `canonical-512-768/<shoe>_canonical/mesh` and `mesh_watertight`, and only falls back to `model.pt` export if those files are missing.


In [ ]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import matplotlib.pyplot as plt

from foot_prior import (
    FootAlignment,
    FootAlignmentConfig,
    FootSDFGrid,
    MeshData,
    build_alignment_from_meshes,
    classify_shoe_points,
    detect_shoe_opening_boundary,
    find_boundary_components,
    get_single_boundary_loop,
    load_triangle_mesh,
    make_supr_to_shoe_axis_remap,
    mesh_bounds,
    principal_yaw_degrees,
    query_foot_sdf_in_shoe_space,
    sole_block_masks,
)
import foot_prior.foot_alignment as _foot_alignment

summary = json.loads(summary_path.read_text())
saved_alignment = FootAlignment.from_json(shoe_out / 'alignment.json')
raw_foot_mesh = load_triangle_mesh(FOOT_OBJ_PATH)
open_mesh_path = Path(summary['outputs']['shoe_open_mesh_obj'])
watertight_mesh_path = Path(summary['outputs']['shoe_watertight_mesh_obj'])
open_mesh = load_triangle_mesh(open_mesh_path)
watertight_mesh = load_triangle_mesh(watertight_mesh_path)
saved_foot_mesh = load_triangle_mesh(shoe_out / 'foot_aligned.obj')
foot_sdf = FootSDFGrid.from_npz(str(FOOT_SDF_PATH), device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

print('raw SUPR foot vertices/faces  :', raw_foot_mesh.vertices.shape, raw_foot_mesh.faces.shape)
print('open shoe vertices/faces      :', open_mesh.vertices.shape, open_mesh.faces.shape)
print('watertight shoe vertices/faces:', watertight_mesh.vertices.shape, watertight_mesh.faces.shape)
print('mesh source                  :', summary.get('mesh_source', 'unknown'))
print('open mesh path               :', open_mesh_path)
print('watertight mesh path         :', watertight_mesh_path)
print('saved aligned foot vertices   :', saved_foot_mesh.vertices.shape)
print('saved scale                   :', saved_alignment.scale)
print('saved plantar_z               :', saved_alignment.plantar_z)
print('saved opening detected        :', summary['opening']['detected'])


## 5. Plot Helpers

This uses the same display convention as the earlier alignment notebook:

```text
display X = raw +X length
display Y = raw +Z width
display Z = raw -Y opening/up
```

So raw `+Y base/bottom` appears downward in the display frame.


In [ ]:
def display_axis_order(alignment=None):
    cfg = alignment.config if alignment is not None else FootAlignmentConfig()
    return [cfg.shoe_length_axis, cfg.shoe_width_axis, cfg.shoe_up_axis]


def display_axis_signs(alignment=None):
    cfg = alignment.config if alignment is not None else FootAlignmentConfig()
    return np.asarray([cfg.shoe_length_sign, cfg.shoe_width_sign, cfg.shoe_up_sign], dtype=np.float32)


def to_display_coords(vertices, alignment=None):
    vertices = np.asarray(vertices, dtype=np.float32)
    return vertices[:, display_axis_order(alignment)] * display_axis_signs(alignment)[None, :]


def to_display_vector(vector, alignment=None):
    vector = np.asarray(vector, dtype=np.float32).reshape(1, 3)
    return to_display_coords(vector, alignment)[0]


def set_axes_equal(ax, vertices_list):
    all_vertices = np.concatenate([np.asarray(v).reshape(-1, 3) for v in vertices_list if np.asarray(v).size], axis=0)
    mn = all_vertices.min(axis=0)
    mx = all_vertices.max(axis=0)
    center = (mn + mx) * 0.5
    radius = max(float((mx - mn).max()) * 0.55, 1e-6)
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_zlim(center[2] - radius, center[2] + radius)
    try:
        ax.set_box_aspect((1, 1, 1))
        ax.set_proj_type('ortho')
    except Exception:
        pass


def sample_faces(mesh, max_faces=3500):
    faces = mesh.faces
    if faces.shape[0] <= max_faces:
        return faces
    idx = np.linspace(0, faces.shape[0] - 1, max_faces).astype(np.int64)
    return faces[idx]


def add_mesh(ax, mesh, color='#cccccc', alpha=0.35, max_faces=3500, linewidth=0.04, alignment=None):
    if mesh.vertices.shape[0] == 0 or mesh.faces.shape[0] == 0:
        return np.empty((0, 3), dtype=np.float32)
    faces = sample_faces(mesh, max_faces=max_faces)
    verts = to_display_coords(mesh.vertices, alignment)
    poly = Poly3DCollection(verts[faces], alpha=alpha, linewidths=linewidth)
    poly.set_facecolor(color)
    poly.set_edgecolor('#222222')
    ax.add_collection3d(poly)
    return verts


def add_arrow(ax, origin_raw, direction_raw, label, color, length=0.06, alignment=None, linewidth=2.5):
    origin = to_display_coords(np.asarray(origin_raw, dtype=np.float32).reshape(1, 3), alignment)[0]
    direction_raw = np.asarray(direction_raw, dtype=np.float32)
    norm = max(float(np.linalg.norm(direction_raw)), 1e-8)
    direction_display = to_display_vector(direction_raw / norm, alignment) * length
    ax.quiver(origin[0], origin[1], origin[2], direction_display[0], direction_display[1], direction_display[2], color=color, linewidth=linewidth)
    tip = origin + direction_display
    if label:
        ax.text(tip[0], tip[1], tip[2], label, color=color, fontsize=9)


def add_shoe_axis_arrows(ax, mesh, alignment=None):
    center = mesh_bounds(mesh.vertices)[3]
    length = float(mesh_bounds(mesh.vertices)[2].max() * 0.34)
    add_arrow(ax, center, [1, 0, 0], '+X length/toe', '#d7191c', length=length, alignment=alignment)
    add_arrow(ax, center, [0, 0, 1], '+Z width', '#2c7bb6', length=length, alignment=alignment)
    add_arrow(ax, center, [0, -1, 0], '-Y opening/up', '#1a9641', length=length, alignment=alignment)
    add_arrow(ax, center, [0, 1, 0], '+Y base/bottom', '#1b9e77', length=length, alignment=alignment)


def plot_scene(alignment, foot_mesh, title, elev=10, azim=-70, show_open=True, show_watertight=True, show_sole=False, sole_mesh=None, show_opening=True, show_anatomy=False):
    fig = plt.figure(figsize=(9, 6))
    ax = fig.add_subplot(111, projection='3d')
    bounds = []
    if show_watertight:
        bounds.append(add_mesh(ax, watertight_mesh, color='#bdbdbd', alpha=0.16, max_faces=4500, linewidth=0.04, alignment=alignment))
    if show_open:
        bounds.append(add_mesh(ax, open_mesh, color='#74add1', alpha=0.22, max_faces=4500, linewidth=0.04, alignment=alignment))
    if show_sole and sole_mesh is not None:
        bounds.append(add_mesh(ax, sole_mesh, color='#d7191c', alpha=0.88, max_faces=9000, linewidth=0.06, alignment=alignment))
    bounds.append(add_mesh(ax, foot_mesh, color='#fdae6b', alpha=0.58, max_faces=1000, linewidth=0.12, alignment=alignment))
    add_shoe_axis_arrows(ax, watertight_mesh, alignment=alignment)

    if show_opening:
        opening = detect_shoe_opening_boundary(open_mesh, alignment.config)
        if opening is not None:
            verts = open_mesh.vertices[np.asarray(opening['vertices'], dtype=np.int64)]
            verts_display = to_display_coords(verts, alignment)
            ax.scatter(verts_display[:, 0], verts_display[:, 1], verts_display[:, 2], s=9, color='#7b3294', depthshade=False)

    if show_anatomy:
        center = mesh_bounds(foot_mesh.vertices)[3]
        linear = alignment.foot_to_shoe[:3, :3]
        arrow_len = float(mesh_bounds(watertight_mesh.vertices)[2].max() * 0.24)
        add_arrow(ax, center, linear @ np.asarray([0, 0, 1], dtype=np.float32), 'SUPR toes', '#f46d43', length=arrow_len, alignment=alignment)
        add_arrow(ax, center, linear @ np.asarray([0, 1, 0], dtype=np.float32), 'SUPR ankle cut', '#7b3294', length=arrow_len, alignment=alignment)
        add_arrow(ax, center, linear @ np.asarray([0, -1, 0], dtype=np.float32), 'SUPR sole', '#3288bd', length=arrow_len, alignment=alignment)

    set_axes_equal(ax, [b for b in bounds if b.shape[0] > 0])
    ax.view_init(elev=elev, azim=azim)
    ax.set_axis_off()
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


## 6. Raw SUPR Foot Axes

This answers: before any transformation, which SUPR coordinate direction points to toes, ankle/up, sole, and width?

```text
SUPR +Z = toes
SUPR +Y = ankle/up
SUPR -Y = sole
SUPR +X = width
```


In [ ]:
def plot_raw_supr_foot(elev=12, azim=-55):
    # This plot intentionally uses raw SUPR coordinates, not the shoe display frame.
    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    faces = sample_faces(raw_foot_mesh, max_faces=1200)
    poly = Poly3DCollection(raw_foot_mesh.vertices[faces], alpha=0.78, linewidths=0.08)
    poly.set_facecolor('#cfcfcf')
    poly.set_edgecolor('#222222')
    ax.add_collection3d(poly)

    center = mesh_bounds(raw_foot_mesh.vertices)[3]
    length = float(mesh_bounds(raw_foot_mesh.vertices)[2].max() * 0.38)

    def raw_arrow(direction, label, color):
        direction = np.asarray(direction, dtype=np.float32)
        direction = direction / max(float(np.linalg.norm(direction)), 1e-8) * length
        ax.quiver(center[0], center[1], center[2], direction[0], direction[1], direction[2], color=color, linewidth=2.5)
        tip = center + direction
        ax.text(tip[0], tip[1], tip[2], label, color=color, fontsize=9)

    raw_arrow([0, 0, 1], '+Z toes', '#d7191c')
    raw_arrow([0, 1, 0], '+Y ankle/up', '#1a9641')
    raw_arrow([0, -1, 0], '-Y sole', '#3288bd')
    raw_arrow([1, 0, 0], '+X width', '#2c7bb6')

    set_axes_equal(ax, [raw_foot_mesh.vertices])
    ax.view_init(elev=elev, azim=azim)
    ax.set_axis_off()
    ax.set_title('Raw SUPR foot coordinate directions')
    plt.tight_layout()
    plt.show()

plot_raw_supr_foot()


## 7. Raw GShell Shoe Axes And Boundary Components

This is the correction that matters most for the shoe:

```text
raw +X = length / toe direction
raw +Y = base or bottom direction
raw -Y = opening/up direction
raw  Z = width direction
```

The open mesh may have a huge missing-base boundary. That is not the ankle/collar opening.


In [ ]:
cfg = FootAlignmentConfig()
print('Raw shoe coordinate colors:')
print('  red   = +X length/toe')
print('  green = -Y opening/up and +Y base/bottom')
print('  blue  = +Z width')
print()
print('raw open shoe bounds      :', mesh_bounds(open_mesh.vertices)[:3])
print('raw watertight shoe bounds:', mesh_bounds(watertight_mesh.vertices)[:3])

components, _ = find_boundary_components(open_mesh)
shoe_min, shoe_max, shoe_size, _ = mesh_bounds(open_mesh.vertices)
signed_up = _foot_alignment.signed_axis_values(open_mesh.vertices, cfg.shoe_up_axis, cfg.shoe_up_sign)
signed_up_min = float(signed_up.min())
signed_up_extent = max(float(signed_up.max() - signed_up_min), 1e-8)

boundary_rows = []
for idx, comp in enumerate(components):
    verts = open_mesh.vertices[np.asarray(comp, dtype=np.int64)]
    bmin, bmax, size, center = mesh_bounds(verts)
    length_ratio = float(size[cfg.shoe_length_axis] / max(shoe_size[cfg.shoe_length_axis], 1e-8))
    width_ratio = float(size[cfg.shoe_width_axis] / max(shoe_size[cfg.shoe_width_axis], 1e-8))
    signed_center_up = float(_foot_alignment.axis_sign(cfg.shoe_up_sign) * center[cfg.shoe_up_axis])
    height_position = float((signed_center_up - signed_up_min) / signed_up_extent)
    sole_like = length_ratio > cfg.opening_max_length_ratio and height_position < cfg.opening_min_height_position
    boundary_rows.append((len(comp), idx, np.round(center, 5), np.round(size, 5), height_position, length_ratio, width_ratio, sole_like))

print('Largest raw open-mesh boundary components:')
for row in sorted(boundary_rows, reverse=True)[:8]:
    count, idx, center, size, height_pos, length_ratio, width_ratio, sole_like = row
    label = 'SOLE-LIKE / missing base' if sole_like else 'possible opening/hole'
    print(f'  component {idx:2d}: n={count:4d} center={center} size={size} height_pos={height_pos:.3f} length_ratio={length_ratio:.3f} width_ratio={width_ratio:.3f} -> {label}')

opening = detect_shoe_opening_boundary(open_mesh, cfg)
print('\nselected opening:', None if opening is None else {'index': opening['index'], 'vertices': int(len(opening['vertices'])), 'center': np.round(opening['center'], 5).tolist()})

plot_scene(saved_alignment, saved_foot_mesh, 'Raw shoe axes, with detected opening in purple', elev=12, azim=-70, show_open=True, show_watertight=True, show_anatomy=False)


## 8. SUPR-To-Shoe Axis Remap Matrix

This is the actual axis conversion before scaling/translation.


In [ ]:
remap = make_supr_to_shoe_axis_remap(
    shoe_length_axis=0,
    shoe_up_axis=1,
    shoe_width_axis=2,
    shoe_length_sign=1.0,
    shoe_up_sign=-1.0,
    shoe_width_sign=1.0,
)
print('SUPR-to-shoe remap matrix:')
print(remap)
print()
for label, vector in {
    'SUPR +X width': np.asarray([1, 0, 0], dtype=np.float32),
    'SUPR +Y ankle/up': np.asarray([0, 1, 0], dtype=np.float32),
    'SUPR +Z toes': np.asarray([0, 0, 1], dtype=np.float32),
    'SUPR -Y sole': np.asarray([0, -1, 0], dtype=np.float32),
}.items():
    print(f'{label:18s} -> shoe {remap @ vector}')
print('\nExpected: toes -> +X, ankle/up -> -Y, sole -> +Y, width -> +Z')


## 9. Build Alignment Live

This mirrors `foot_alignment_playground.ipynb`: the notebook computes the alignment directly from the raw SUPR foot and shoe meshes.

The saved batch alignment should match the default values here.


In [ ]:
def make_alignment(
    length_ratio=0.78,
    scale_multiplier=1.0,
    plantar_clearance=0.032,
    plantar_band=0.012,
    surface_band=0.005,
    clearance=0.005,
    ankle_radius=0.025,
    auto_yaw=True,
    align_ankle_to_opening=True,
    yaw_degrees=0.0,
    pitch_degrees=0.0,
    roll_degrees=0.0,
    tx=0.0,
    ty=0.0,
    tz=0.0,
):
    config = FootAlignmentConfig(
        length_ratio=length_ratio,
        scale_multiplier=scale_multiplier,
        plantar_clearance=plantar_clearance,
        plantar_band=plantar_band,
        surface_band=surface_band,
        clearance=clearance,
        ankle_radius=ankle_radius,
        shoe_length_axis=0,
        shoe_up_axis=1,
        shoe_width_axis=2,
        shoe_length_sign=1.0,
        shoe_up_sign=-1.0,
        shoe_width_sign=1.0,
        align_ankle_to_opening=align_ankle_to_opening,
        auto_yaw=auto_yaw,
        yaw_degrees=yaw_degrees,
        pitch_degrees=pitch_degrees,
        roll_degrees=roll_degrees,
        translation_offset=(tx, ty, tz),
    )
    return build_alignment_from_meshes(
        foot_mesh=raw_foot_mesh,
        shoe_mesh=watertight_mesh,
        opening_mesh=open_mesh,
        config=config,
    )

live_alignment = make_alignment()
live_foot_mesh = MeshData(live_alignment.transform_foot_to_shoe(raw_foot_mesh.vertices), raw_foot_mesh.faces)
print('shoe principal yaw degrees in x-z length/width plane:', principal_yaw_degrees(watertight_mesh.vertices, length_axis=0, width_axis=2))
print('live scale:', live_alignment.scale)
print('saved scale:', saved_alignment.scale)
print('live opening center:', live_alignment.opening_center)
print('saved opening center:', saved_alignment.opening_center)
plot_scene(live_alignment, live_foot_mesh, 'Default live alignment: foot inside shoe', elev=10, azim=-70, show_anatomy=True)


## 10. Interactive Alignment Sliders

Use this when a shoe looks wrong. The goal is not a pretty picture; the goal is anatomical plausibility:

```text
toes inside toe box
heel/ankle near the opening side
foot bottom above the sole/base
foot not obviously outside the shoe walls
```


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    length_w = widgets.FloatSlider(value=0.78, min=0.55, max=1.05, step=0.01, description='length')
    scale_w = widgets.FloatSlider(value=1.0, min=0.65, max=1.30, step=0.01, description='scale')
    plantar_w = widgets.FloatSlider(value=0.032, min=0.002, max=0.060, step=0.002, description='plantar')
    auto_yaw_w = widgets.Checkbox(value=True, description='auto yaw')
    align_opening_w = widgets.Checkbox(value=True, description='ankle opening')
    tx_w = widgets.FloatSlider(value=0.0, min=-0.10, max=0.10, step=0.002, description='tx')
    ty_w = widgets.FloatSlider(value=0.0, min=-0.10, max=0.10, step=0.002, description='ty')
    tz_w = widgets.FloatSlider(value=0.0, min=-0.10, max=0.10, step=0.002, description='tz')
    yaw_w = widgets.FloatSlider(value=0.0, min=-45, max=45, step=1.0, description='yaw')
    pitch_w = widgets.FloatSlider(value=0.0, min=-35, max=35, step=1.0, description='pitch')
    roll_w = widgets.FloatSlider(value=0.0, min=-35, max=35, step=1.0, description='roll')
    elev_w = widgets.IntSlider(value=10, min=-90, max=90, step=2, description='elevation')
    azim_w = widgets.IntSlider(value=-70, min=-180, max=180, step=5, description='azimuth')
    show_open_w = widgets.Checkbox(value=True, description='open mesh')
    show_wat_w = widgets.Checkbox(value=True, description='watertight')
    show_anatomy_w = widgets.Checkbox(value=True, description='anatomy arrows')
    out = widgets.Output()

    def current_alignment():
        return make_alignment(
            length_ratio=length_w.value,
            scale_multiplier=scale_w.value,
            plantar_clearance=plantar_w.value,
            auto_yaw=auto_yaw_w.value,
            align_ankle_to_opening=align_opening_w.value,
            yaw_degrees=yaw_w.value,
            pitch_degrees=pitch_w.value,
            roll_degrees=roll_w.value,
            tx=tx_w.value,
            ty=ty_w.value,
            tz=tz_w.value,
        )

    def redraw(_=None):
        aln = current_alignment()
        foot = MeshData(aln.transform_foot_to_shoe(raw_foot_mesh.vertices), raw_foot_mesh.faces)
        with out:
            clear_output(wait=True)
            print(f'scale={aln.scale:.5f}, plantar_z={aln.plantar_z:.5f}, auto_yaw={aln.auto_yaw_degrees:.2f}')
            print(f'opening center={aln.opening_center}')
            print('orange=foot, gray=watertight, blue=open mesh, purple=detected opening')
            plot_scene(
                aln,
                foot,
                'interactive foot alignment',
                elev=elev_w.value,
                azim=azim_w.value,
                show_open=show_open_w.value,
                show_watertight=show_wat_w.value,
                show_anatomy=show_anatomy_w.value,
            )

    controls = widgets.VBox([
        widgets.HBox([length_w, scale_w, plantar_w, auto_yaw_w, align_opening_w]),
        widgets.HBox([tx_w, ty_w, tz_w]),
        widgets.HBox([yaw_w, pitch_w, roll_w]),
        widgets.HBox([elev_w, azim_w, show_open_w, show_wat_w, show_anatomy_w]),
    ])
    for w in [length_w, scale_w, plantar_w, auto_yaw_w, align_opening_w, tx_w, ty_w, tz_w, yaw_w, pitch_w, roll_w, elev_w, azim_w, show_open_w, show_wat_w, show_anatomy_w]:
        w.observe(redraw, names='value')
    display(controls, out)
    redraw()
except Exception as exc:
    print('ipywidgets are not available; showing static default alignment.')
    print(type(exc).__name__ + ':', exc)
    plot_scene(live_alignment, live_foot_mesh, 'default live alignment', show_anatomy=True)


## 11. SDF Region Classification

This checks shoe vertices against the aligned foot SDF.

```text
SDF < 0       = point is inside the foot volume
SDF near 0    = point is near the foot surface
below plantar = point is on the base/material side of the foot bottom
```


In [ ]:
loop = get_single_boundary_loop(raw_foot_mesh)
ankle_loop_shoe = live_alignment.transform_foot_to_shoe(raw_foot_mesh.vertices[loop])
watertight_regions = classify_shoe_points(watertight_mesh.vertices, foot_sdf, live_alignment, ankle_loop_shoe)

for key in ['inside_foot', 'near_foot_surface', 'clearance_violation', 'below_plantar', 'near_ankle']:
    mask = np.asarray(watertight_regions[key], dtype=bool)
    print(f'{key:22s}: {int(mask.sum()):5d} / {mask.shape[0]} = {float(mask.mean()):.4f}')
print('SDF min/mean/max:', float(watertight_regions['sdf'].min()), float(watertight_regions['sdf'].mean()), float(watertight_regions['sdf'].max()))


## 12. Sole-Block Region: Old Support Mask Vs New Footprint Mask

The earlier notebook used this rough support idea:

```text
below plantar plane AND foot_sdf <= 0.035
```

The new dataset script uses the more explicit sole-block idea:

```text
below plantar plane AND under the projected foot footprint
```

The red surface below is the newer triangle/surface mask.


In [ ]:
PLANTAR_SDF_BAND = 0.035

centroids = watertight_mesh.vertices[watertight_mesh.faces].mean(axis=1)
centroid_sdf = query_foot_sdf_in_shoe_space(centroids, foot_sdf, live_alignment)
cfg = live_alignment.config
centroid_signed_up = _foot_alignment.signed_axis_values(centroids, cfg.shoe_up_axis, cfg.shoe_up_sign)
threshold_signed_up = _foot_alignment.axis_sign(cfg.shoe_up_sign) * live_alignment.plantar_z + cfg.plantar_band
old_face_support = (centroid_signed_up <= threshold_signed_up) & (centroid_sdf <= PLANTAR_SDF_BAND)

sole_masks = sole_block_masks(
    watertight_mesh,
    live_foot_mesh.vertices,
    live_alignment,
    footprint_margin=0.012,
    plantar_band=cfg.plantar_band,
)
new_face_support = sole_masks['face_sole_block']

print('old SDF-band support faces :', int(old_face_support.sum()), '/', watertight_mesh.faces.shape[0], '=', float(old_face_support.mean()))
print('new footprint support faces:', int(new_face_support.sum()), '/', watertight_mesh.faces.shape[0], '=', float(new_face_support.mean()))

new_sole_mesh = MeshData(watertight_mesh.vertices, watertight_mesh.faces[new_face_support])
plot_scene(live_alignment, live_foot_mesh, 'New sole-block surface: below foot + under foot footprint', elev=12, azim=-70, show_open=False, show_watertight=True, show_sole=True, sole_mesh=new_sole_mesh, show_anatomy=False)


## 13. Saved Batch PNG/PLY Outputs For This Shoe

These are the files generated by the batch script. Open the PLY/OBJ files in MeshLab when the notebook view is still confusing.


In [ ]:
for key, path in summary['outputs'].items():
    print(f'{key:32s}: {path}')
print('\nextra output:')
print('axis_alignment_views_png      :', shoe_out / 'axis_alignment_views.png')


## 14. Batch Summary Table

After running the full script, this table tells us which shoes succeeded and how large the sole-block region was.


In [ ]:
summary_csv = DEBUG_ROOT / 'summary.csv'
print('summary csv:', summary_csv)
if summary_csv.exists():
    with summary_csv.open('r') as f:
        rows = list(csv.DictReader(f))
    print('rows:', len(rows))
    for row in rows:
        print(
            row['status'].ljust(10),
            row['shoe_name'],
            'opening=', row.get('opening_detected', ''),
            'sole_face_fraction=', row.get('sole_face_fraction', ''),
        )
else:
    print('No summary.csv yet. Run the full-dataset command in section 2.')


## 15. Run Section 1.4 V4 Hybrid Optimizer

This runs the hybrid foot placement optimizer using the trained turntable meshes, V4 support-footbed analysis, the baseline alignment outputs from this notebook, and the previous translation-friendly optimizer as a warm start when available.


In [ ]:
import runpy
import shlex

# Safety guard: set this to True when you actually want to regenerate optimized fit artifacts.
RUN_FIT_OPTIMIZATION = True

# Set FIT_OPTIMIZE_ALL_SCENES=True to process every trained scene under BASELINE_SUBDIR.
# Set it to False and list scene names if you only want a few examples.
FIT_OPTIMIZE_ALL_SCENES = True
FIT_OPTIMIZE_SCENE_NAMES = [
    'Adidas-Yeezy-Boost-350-V2-Desert-Sage-Infant_turntable',
    'Crocs-Classic-Clog-Cinnamon-Toast-Crunch-Gs_turntable',
    'Ugg-Classic-Short-Ii-Boot-Rock-Rose-Toddler_turntable',
    'Air-Jordan-1-Retro-High-Og-White-Cement-Gs_turntable',
]

FIT_OPTIMIZE_OVERWRITE = True
FIT_OPTIMIZE_DEVICE = 'cuda'
FIT_OPTIMIZE_ADAM_STEPS = 120
FIT_OPTIMIZE_LBFGS_STEPS = 12

TURN_TABLE_MESH_ROOT = BASELINE_OUTPUT_ROOT / BASELINE_SUBDIR
SUPPORT_FOOTBED_ROOT = BASELINE_OUTPUT_ROOT / 'support_footbed_analysis_v4'
WARM_START_ALIGNMENT_ROOT = BASELINE_OUTPUT_ROOT / 'foot_alignment_optimized_turntable-512-768'
OPTIMIZED_ALIGNMENT_ROOT = BASELINE_OUTPUT_ROOT / 'foot_alignment_optimized_v4_hybrid_turntable-512-768'

fit_optimizer_script = FOOTSHELL_ROOT / 'scripts' / 'run_foot_fit_optimization.py'
fit_optimizer_argv = [
    str(fit_optimizer_script),
    '--mesh-root', str(TURN_TABLE_MESH_ROOT),
    '--support-root', str(SUPPORT_FOOTBED_ROOT),
    '--baseline-alignment-root', str(DEBUG_ROOT),
    '--warm-start-alignment-root', str(WARM_START_ALIGNMENT_ROOT),
    '--style-mode', 'auto',
    '--output-root', str(OPTIMIZED_ALIGNMENT_ROOT),
    '--foot-obj', str(FOOT_OBJ_PATH),
    '--device', FIT_OPTIMIZE_DEVICE,
    '--adam-steps', str(FIT_OPTIMIZE_ADAM_STEPS),
    '--lbfgs-steps', str(FIT_OPTIMIZE_LBFGS_STEPS),
]
if not FIT_OPTIMIZE_ALL_SCENES:
    for scene_name in FIT_OPTIMIZE_SCENE_NAMES:
        fit_optimizer_argv += ['--scene', scene_name]
if FIT_OPTIMIZE_OVERWRITE:
    fit_optimizer_argv += ['--overwrite']

shell_equivalent = [str(GSHELL_ENV / 'bin' / 'python')] + fit_optimizer_argv
print('CUDA_VISIBLE_DEVICES=' + os.environ.get('CUDA_VISIBLE_DEVICES', ''))
print(' '.join(shlex.quote(part) for part in shell_equivalent))
print('optimized output root:', OPTIMIZED_ALIGNMENT_ROOT)

if RUN_FIT_OPTIMIZATION:
    old_argv = sys.argv[:]
    try:
        sys.argv = fit_optimizer_argv
        runpy.run_path(str(fit_optimizer_script), run_name='__main__')
    finally:
        sys.argv = old_argv
else:
    print('Set RUN_FIT_OPTIMIZATION = True, then execute this cell to run run_foot_fit_optimization.py.')
